# 2D → 3D Pipeline — Colab **Run all** (P6)

**Runtime ▸ Change runtime type ▸ T4 GPU** → **Runtime ▸ Run all**. Xong, không cần bấm gì thêm.

| Cell | Việc |
| :--: | :--- |
| 1 | Clone repo + cài dependencies + **DUSt3R thật (P2)** + TripoSR |
| 2 | Tự tải 5 ảnh test **Google Scanned Objects** (vật thật, nhiều góc) — không cần upload |
| 3 | Bật FastAPI + Cloudflare Tunnel → in URL công khai |
| 4 | Chạy thử pipeline + **in TOÀN BỘ log P1→P5** |

Run all xong: mở URL ở Cell 3 → kéo 4–8 ảnh vào Web UI.

> ⚠️ Nếu Cell 1 báo `DUSt3R chưa import được` thì P2 rơi về MOCK — **hình dạng là NGẪU NHIÊN**.
> Đừng chụp kết quả, copy log Cell 4 gửi lại.


In [ ]:
# Cell 1: Clone repo (branch test) + deps + DUSt3R thật (P2) + TripoSR
# Colab đã có torch GPU sẵn — KHÔNG cài lại (đè bản Colab -> vỡ CUDA runtime).
import os, sys, shutil

os.chdir('/content')   # kernel có thể đang đứng trong thư mục đã bị xoá -> mọi lệnh ! sau đó fail 'getcwd'

REPO, BRANCH = '/content/Img2d-to-3d', 'P6-dust3r-real-test'
if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)      # dọn thư mục rỗng còn sót từ lần chạy trước
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} pull -q
os.chdir(REPO)
!git log --oneline -1
!ls notebook/backend/app.py

!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx scikit-image opencv-python-headless kornia transformers xatlas

# ── DUSt3R thật (P2): hình dạng 3D lấy từ ảnh ──
if not os.path.isdir('/content/dust3r/.git'):
    shutil.rmtree('/content/dust3r', ignore_errors=True)
    !git clone -q --recursive https://github.com/naver/dust3r.git /content/dust3r
# KHÔNG chạy requirements.txt của dust3r: nó kéo torch/gradio/tensorboard, đè bản Colab -> vỡ CUDA
!pip install -q roma tqdm matplotlib einops safetensors

sys.path.append('/content/dust3r')
try:
    import dust3r  # noqa: F401
    print('✅ DUSt3R OK -> P2 chạy THẬT (hình dạng lấy từ ảnh)')
except Exception as e:
    print('⚠️ DUSt3R chưa import được -> P2 MOCK (hình dạng NGẪU NHIÊN):', e)

# ── TripoSR (chỉ cần cho chế độ 1 ảnh / nhánh Quality FAIL) ──
if not os.path.isdir('/content/TripoSR/.git'):
    shutil.rmtree('/content/TripoSR', ignore_errors=True)
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
# KHÔNG chạy requirements.txt của TripoSR: ghim transformers==4.35.0 (đè bản RMBG-2.0 cần)
!pip install -q omegaconf einops imageio
!pip install -q torchmcubes 2>/dev/null || echo 'torchmcubes thiếu (Colab là Python 3.13) -> TripoSR tự CPU fallback'
os.system(f'rm -rf {REPO}/notebook/backend/tsr && cp -r /content/TripoSR/tsr {REPO}/notebook/backend/tsr')

# ── RMBG-2.0 (tách nền) — repo GATED, cần token HuggingFace ──
# Không có token thì P1 trả mask toàn 1 (không tách nền); pipeline vẫn chạy bình thường.
# Đã ĐO trên 5 ảnh GSO (nền trơn): mask chỉ đổi 4% thể tích mesh, vì P4 đã tự lọc nền
# bằng ngưỡng confidence của DUSt3R. Nền càng rối thì mask càng có ích.
# Bật (1 lần): 🔑 sidebar trái Colab -> Add new secret | Name: HF_TOKEN | Value: token của bạn
#              -> bật "Notebook access". Và bấm Agree ở https://huggingface.co/briaai/RMBG-2.0
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN đã nạp -> P1 tách nền thật bằng RMBG-2.0')
except Exception:
    print('ℹ️ Không có HF_TOKEN -> P1 trả mask toàn 1 (không tách nền). Vẫn chạy bình thường.')

print('Deps OK | cwd =', os.getcwd())


In [ ]:
# Cell 2: Tự tải 5 ảnh test Google Scanned Objects — vật THẬT, nhiều góc, có ground truth
# (1030 vật / 17 nhóm). Không cần upload tay.
import os, json, shutil

os.chdir('/content')
!pip install -q datasets
from datasets import load_dataset

OBJ_INDEX = 0   # đổi số này để thử vật khác. Xem danh sách 40 vật đầu ở cell cuối.

ds = load_dataset("suvadityamuk/google-scanned-objects", split="train", streaming=True)
sample = next((s for i, s in enumerate(ds) if i == OBJ_INDEX), None)
assert sample is not None, f"Không có vật index {OBJ_INDEX}"

meta = sample["json"]
if isinstance(meta, str):
    meta = json.loads(meta)
print(f"Vật #{OBJ_INDEX}: {meta.get('name')} | nhóm: {meta.get('category') or meta.get('category_name')}")
print("json keys:", list(meta)[:15])

OUT = '/content/Img2d-to-3d/data/input/gso'   # Cell 4 đọc đúng thư mục này
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

# key giữ nguyên ĐUÔI file: thumbnail_0.jpg ... thumbnail_4.jpg
thumbs = sorted(k for k in sample if k.startswith("thumbnail"))
for i, k in enumerate(thumbs[:5], 1):
    sample[k].convert("RGB").save(f"{OUT}/view_{i:02d}.jpg", quality=95)   # 640x460, 5 góc quanh vật
print(f"Đã lưu {min(len(thumbs), 5)} ảnh:", sorted(os.listdir(OUT)))

try:                        # ground truth để so sánh kết quả (không bắt buộc)
    sample["glb"].export(f"{OUT}/gt.glb")
    print("GT:", f"{OUT}/gt.glb")
except Exception as e:
    print("(bỏ qua ground truth:", type(e).__name__, ")")

os.chdir('/content/Img2d-to-3d')
print("cwd =", os.getcwd())


In [ ]:
# Cell 3: Bật FastAPI + Cloudflare Tunnel (KHÔNG cần tài khoản Cloudflare)
import subprocess, time, os, re, urllib.request

REPO    = '/content/Img2d-to-3d'                            # tuyệt đối, KHÔNG phụ thuộc cwd
BACKEND = os.path.join(REPO, 'notebook', 'backend')         # app.py tạo temp_uploads/ & outputs/ theo cwd
assert os.path.isdir(BACKEND), f'Không thấy {BACKEND} -> chạy lại Cell 1'
os.chdir(REPO)

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

env = dict(os.environ)
env['PYTHONPATH'] = '/content/dust3r' + os.pathsep + env.get('PYTHONPATH', '')
# Nút vặn chất lượng — đã ĐO bằng Chamfer distance tới ground truth:
#   TSDF_RES=256 -> LỆCH hơn 128 (+11%). 256 chỉ trông mượt hơn, KHÔNG chính xác hơn -> để 128.
#   DUST3R_NITER: 100 -> 0.16067 | 300 -> 0.17599 (ít vòng lại chính xác hơn, nhưng 300 an toàn hơn)
env.setdefault('TSDF_RES', '128')
env.setdefault('DUST3R_NITER', '300')

# Log ra FILE, KHÔNG dùng PIPE: pipe không ai đọc -> đầy 64KB -> tiến trình con TẮC giữa chừng.
server = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT, text=True,
)

# Chờ server sẵn sàng (health check, không sleep mù). Lần đầu nạp model: DUSt3R ~2.3GB -> vài phút.
ready = False
for _ in range(150):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            print('Server READY:', r.read().decode()); ready = True; break
    except Exception:
        time.sleep(2)
if not ready:
    print('Server KHÔNG khởi động. Log cuối:')
    print(open('/content/server.log', encoding='utf-8', errors='replace').read()[-4000:])

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT, text=True,
)
url = None
for _ in range(60):                     # đọc FILE, không đọc pipe -> cloudflared không bao giờ tắc
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com',
                  open('/content/tunnel.log', encoding='utf-8', errors='replace').read())
    if m:
        url = m.group(0); print('\n🌐 WEB UI  :', url); print('📘 SWAGGER :', url + '/docs'); break
    time.sleep(1)
if url is None:
    print('Không lấy được URL tunnel — xem /content/tunnel.log')


In [ ]:
# Cell 4: Chạy thử pipeline + in TOÀN BỘ log P1→P5
# Nếu có lỗi: copy nguyên phần "LOG SERVER" bên dưới gửi lại.
import glob, subprocess, json, os

os.chdir('/content/Img2d-to-3d')

gso = sorted(glob.glob('/content/Img2d-to-3d/data/input/gso/view_*.jpg'))
mv  = sorted(glob.glob('/content/Img2d-to-3d/data/input/multi_view/view_*.jpg'))
imgs = gso if len(gso) >= 4 else mv
assert imgs, 'Không có ảnh test -> chạy lại Cell 2'
print(f'Dùng {len(imgs)} ảnh:', [os.path.basename(p) for p in imgs[:8]])
# 4–8 ảnh là vùng tốt nhất. >8 ảnh bị preprocess.py cắt xuống 6 -> TỆ HƠN up 8.

if len(imgs) >= 2:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/']
    for p in imgs[:8]:
        cmd += ['-F', f'files=@{p}']
else:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/single/', '-F', f'file=@{imgs[0]}']

r = subprocess.run(cmd, capture_output=True, text=True)
print('\n--- KẾT QUẢ API ---')
try:
    print(json.dumps(json.loads(r.stdout), indent=2, ensure_ascii=False))
except Exception:
    print('stdout:', r.stdout[:1000], '\nstderr:', r.stderr[-500:])

print('\n--- FILE .glb ---')
for f in sorted(glob.glob('/content/Img2d-to-3d/notebook/backend/outputs/*.glb')):
    print(f'{os.path.basename(f)}  {os.path.getsize(f)} bytes')

print('\n--- LOG SERVER (P1→P5) — copy từ đây nếu cần báo lỗi ---')
print(open('/content/server.log', encoding='utf-8', errors='replace').read()[-8000:])


In [ ]:
# (BỎ COMMENT rồi chạy cell này) — xem 40 vật đầu để chọn OBJ_INDEX cho Cell 2
# from datasets import load_dataset
# ds = load_dataset("suvadityamuk/google-scanned-objects", split="train", streaming=True)
# for i, s in enumerate(ds):
#     if i >= 40: break
#     m = s["json"]
#     print(i, (m.get('category') or m.get('category_name')), '|', m.get('name'))


## Dừng server

**Runtime ▸ Restart session** (hoặc chạy `!pkill -f uvicorn`). File `.glb` nằm ở
`notebook/backend/outputs/` — tải về bằng panel **Files** bên trái.

## Nguồn ảnh khác

| Nguồn | Đặc điểm |
| :--- | :--- |
| **Google Scanned Objects** (Cell 2) | 1030 vật / 17 nhóm, 5 ảnh render + `gt.glb` để so |
| OmniObject3D | 6000 vật / 190 nhóm, **100 ảnh 800×800** mỗi vật (cần đăng ký OpenDataLab) |
| CO3D (Meta) | 19k vật, ảnh chụp thật ngoài đời, có mask tách nền sẵn (bản nhỏ 8.9 GB) |
| DTU | Benchmark MVS, 49–64 góc, có ground-truth point cloud |

**Tốt nhất vẫn là ảnh tự chụp**: đặt vật lên nền trơn → đi vòng quanh, **8 tấm cách đều ~45°**,
giữ nguyên khoảng cách và độ cao máy, **không xoay vật**.
